[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akifkoc/sabanci-qiskitfallfest26/blob/main/2-cok_kubitli_sistemler.ipynb)

# Bölüm 2: Birden Fazla Kubit

**Qiskit ile Kuantum Programlamaya Giriş** · Qiskit Fall Fest 2026 · Sabancı Üniversitesi

Bu bölümün sonunda şunları yapabileceksiniz:

- iki kubitlik bir durumu tansör çarpımıyla yazmak,
- Qiskit'in kubit sırasını doğru okumak,
- CNOT kapısını kullanmak.

**Soru**ları önce kendiniz düşünün, sonra "Cevap"a tıklayın. Takıldığınız yerde bize sorun.

In [ ]:
# Google Colab'da bu hücreyi bir kez çalıştırın (yaklaşık 1 dakika sürer).
# Kendi bilgisayarınızda requirements.txt ile kurulum yaptıysanız bu hücreyi atlayabilirsiniz.
%pip install -q "qiskit[visualization]==2.6.0" "qiskit-aer==0.18.0"

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram

simulator = AerSimulator()

## 1. İki bit ve tansör çarpımı

İki hileli para atalım: birincisi $\begin{pmatrix} 0{,}2 \\ 0{,}8 \end{pmatrix}$, ikincisi $\begin{pmatrix} 0{,}6 \\ 0{,}4 \end{pmatrix}$ durumunda. İki bit birlikte dört durumdan birindedir: $00$, $01$, $10$, $11$ (soldaki karakter birinci para).

**Soru:** $00$ durumunun olasılığı nedir?

<details>
<summary><b>Cevap</b></summary>

Paralar bağımsız olduğu için olasılıklar çarpılır: $0{,}2 \cdot 0{,}6 = 0{,}12$.

</details>

> **Doğrusal cebir: tansör çarpımı**
>
> İki vektörün elemanlarının bütün ikili çarpımlarını sırayla alt alta yazmaya **tansör çarpımı** (*tensor product*) denir:
>
> $$ \begin{pmatrix} a_1 \\ b_1 \end{pmatrix} \otimes \begin{pmatrix} a_2 \\ b_2 \end{pmatrix} = \begin{pmatrix} a_1 a_2 \\ a_1 b_2 \\ b_1 a_2 \\ b_1 b_2 \end{pmatrix} $$
>
> $n$ tane iki elemanlı vektörün tansör çarpımı $2^n$ elemanlıdır.

İki paramız için:

$$ \begin{pmatrix} 0{,}2 \\ 0{,}8 \end{pmatrix} \otimes \begin{pmatrix} 0{,}6 \\ 0{,}4 \end{pmatrix} = \begin{pmatrix} 0{,}2 \cdot 0{,}6 \\ 0{,}2 \cdot 0{,}4 \\ 0{,}8 \cdot 0{,}6 \\ 0{,}8 \cdot 0{,}4 \end{pmatrix} = \begin{pmatrix} 0{,}12 \\ 0{,}08 \\ 0{,}48 \\ 0{,}32 \end{pmatrix} \begin{matrix} 00 \\ 01 \\ 10 \\ 11 \end{matrix} $$

NumPy'da bu hesabı `np.kron` yapar. Adı **Kronecker çarpımından** (*Kronecker product*) gelir: yukarıdaki hesap aslında Kronecker çarpımıdır. Tansör çarpımı daha genel bir kavramdır; vektörleri bu sırayla yazdığımızda ikisi aynı sonucu verir.

## 2. İki kubit

İki kubitin dört baz durumu vardır: $|00\rangle$, $|01\rangle$, $|10\rangle$, $|11\rangle$. Genel durum bunların üst üste binmesidir:

$$ \alpha_{00}|00\rangle + \alpha_{01}|01\rangle + \alpha_{10}|10\rangle + \alpha_{11}|11\rangle $$

Ayrı ayrı iki kubitin ortak durumu yine tansör çarpımıyla bulunur.

**Soru:** $\left(\frac{1}{\sqrt{3}}|0\rangle + \frac{\sqrt{2}}{\sqrt{3}}|1\rangle\right) \otimes \left(-\frac{1}{\sqrt{2}}|0\rangle + \frac{1}{\sqrt{2}}|1\rangle\right)$ nedir? (Çarpma gibi, toplama üzerine dağıtın.)

<details>
<summary><b>Cevap</b></summary>

$$ -\frac{1}{\sqrt{6}}|00\rangle + \frac{1}{\sqrt{6}}|01\rangle - \frac{\sqrt{2}}{\sqrt{6}}|10\rangle + \frac{\sqrt{2}}{\sqrt{6}}|11\rangle $$

Olasılıklar $\frac16$, $\frac16$, $\frac26$, $\frac26$; toplamları $1$.

</details>

**Soru:** 50 kubitlik bir durumu tam olarak yazmak için kaç genlik gerekir?

<details>
<summary><b>Cevap</b></summary>

$2^{50} \approx 10^{15}$. Kubit sayısı arttıkça kuantum durumlarını klasik bilgisayarda saklamak hızla imkânsızlaşır.

</details>

## 3. Qiskit'te kubit sırası

Qiskit sonuçları **sağdan sola** yazar: en sağdaki karakter $q_0$'ın, solundaki $q_1$'in sonucudur. İki kubitlik durumları da bu sırayla, $|q_1 q_0\rangle$ biçiminde yazacağız. (Dikkat: paralarda birinci parayı solda yazmıştık; Qiskit'te $q_0$ en sağdadır.)

Aşağıda 4 kubitten yalnızca $q_2$'ye X uyguluyoruz:

In [ ]:
qc = QuantumCircuit(4)
qc.x(2)
qc.measure_all()   # tüm kubitleri ölç (çizimdeki kesikli çizgiyi de bu ekler)

counts = simulator.run(qc, shots=1000).result().get_counts()
print(counts)
qc.draw("mpl")

**Soru:** Sonuç neden `0100`, yani neden sağdan sola?

<details>
<summary><b>Cevap</b></summary>

Qiskit sonucu bir **ikili sayı** (*binary number*) gibi yazar: en sağdaki basamak $2^0$'ın basamağıdır ve $q_0$ oraya yazılır. `0100` ikili sayı olarak $4 = 2^2$'dir, yani $q_2 = 1$. Qiskit belgeleri bu sıraya *little-endian* der.

</details>

### Görev 1

3 kubitlik bir devrede ölçüm sonucu her zaman `011` olacak şekilde X kapıları uygulayın.

In [ ]:
qc = QuantumCircuit(3)

# kodunuzu buraya yazın

qc.measure_all()
counts = simulator.run(qc, shots=1000).result().get_counts()
print(counts)

In [ ]:
# kontrol
if counts == {"011": 1000}:
    print("DOĞRU")
else:
    print("Henüz değil. İpucu: sonucu sağdan sola okuyun; en sağdaki karakter q0.")

## 4. CNOT kapısı

CNOT (*controlled-NOT*) iki kubitlik bir kapıdır: **kontrol kubiti** $|1\rangle$ ise **hedef kubite** (*target qubit*) X uygulanır, $|0\rangle$ ise hedef değişmez. Qiskit'te `qc.cx(kontrol, hedef)` ile yazılır.

Dört girdiyi deneyelim (kontrol $q_0$, hedef $q_1$; girdiler $q_1 q_0$ sırasıyla):

In [ ]:
for inp in ["00", "01", "10", "11"]:
    qc = QuantumCircuit(2)
    if inp[1] == "1":   # en sağdaki karakter q0
        qc.x(0)
    if inp[0] == "1":   # soldaki karakter q1
        qc.x(1)
    qc.cx(0, 1)         # kontrol: q0, hedef: q1
    qc.measure_all()
    out = list(simulator.run(qc, shots=100).result().get_counts())[0]
    print("girdi:", inp, " ->  çıktı:", out)

**Soru:** Hangi girdilerde çıktı değişti? Neden?

<details>
<summary><b>Cevap</b></summary>

`01` → `11` ve `11` → `01`: yalnızca $q_0$ (sağdaki) $1$ iken $q_1$ (soldaki) çevrildi. $q_0$ hiç değişmez.

</details>

### Görev 2

Bu kez kontrol $q_1$, hedef $q_0$ olsun. Girdi $|q_1 q_0\rangle = |10\rangle$ iken çıktı ne olur? Önce tahmin edin, sonra deneyin.

In [ ]:
qc = QuantumCircuit(2)
qc.x(1)            # girdi: |10⟩

# kodunuzu buraya yazın (kontrol q1, hedef q0)

qc.measure_all()
counts = simulator.run(qc, shots=1000).result().get_counts()
print(counts)
qc.draw("mpl")

## Özet

- Ayrı kubitlerin ortak durumu tansör çarpımıyla bulunur; $n$ kubitin $2^n$ baz durumu vardır.
- Qiskit sonuçları sağdan sola yazar: en sağdaki karakter $q_0$'dır.
- CNOT, kontrol kubiti $1$ ise hedef kubite X uygular: `qc.cx(kontrol, hedef)`.

**Sıradaki bölüm:** [Bölüm 3: Dolaşıklık ve Yoğun Kodlama](https://colab.research.google.com/github/akifkoc/sabanci-qiskitfallfest26/blob/main/3-dolasiklik_ve_yogun_kodlama.ipynb) (H ve CNOT kapılarıyla dolaşıklık ve yoğun kodlama).

### Daha fazlası

- Qiskit: [ibm.com/quantum/qiskit](https://www.ibm.com/quantum/qiskit)
- IBM Quantum Learning: [quantum.cloud.ibm.com/learning](https://quantum.cloud.ibm.com/learning/en)
- QWorld QBook101, Bölüm 2.5: [qworld.net/qbook101](https://qworld.net/qbook101/)
- QTurkey (Kuantum Türkiye) etkinlikleri: [qturkey.org/events](https://qturkey.org/events/)